In [4]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import math
import re

import sys
sys.path.append('../../..')
from mount_drive import mount_s_drive

In [5]:
mount_s_drive(subfolder='LCICM/Databases/eICU')

Password for mbranda1: ········


Mounting S drive subfolder LCICM/Databases/eICU
The following files and folders have been mounted to /home/idies/workspace/SAFE/:
  lab.csv
  respiratoryCharting.csv
  nurseCharting.csv
  patient.csv
  apacheApsVar.csv
  respiratoryCare.csv
  vitalPeriodic.csv
  carePlanEOL.csv
  nurseAssessment.csv
  infusionDrug.csv
  hospital.csv
  nurseCare.csv
  carePlanInfectiousDisease.csv
  physicalExam.csv
  carePlanGoal.csv
  admissionDx.csv
  carePlanCareProvider.csv
  medication.csv
  note.csv
  customLab.csv
  vitalAperiodic.csv
  apachePatientResult.csv
  carePlanGeneral.csv
  intakeOutput.csv
  admissionDrug.csv
  pastHistory.csv
  treatment.csv
  apachePredVar.csv
  allergy.csv
  diagnosis.csv
  microLab.csv


mkdir: cannot create directory '/home/idies/workspace/SAFE/': File exists


In [58]:
myHours = 60 * 12

In [59]:
database_folder = '/home/idies/workspace/SAFE/'

### Patients 

In [60]:
patients_df = pd.read_csv(database_folder+'patient.csv')

In [61]:
myIds = pd.read_csv('../../../Isalis/eICU/eICU_patientunitstayids.txt', header = None)
myIds

,0
0,142723.0
1,145204.0
2,145205.0
3,145830.0
4,146619.0
...,...
3400,3352445.0
3401,3352512.0
3402,3352727.0
3403,3353194.0


In [62]:
patients_df = patients_df[patients_df['patientunitstayid'].isin(myIds[0])]

In [63]:
myPredictorsDf = patients_df[['patientunitstayid', 'gender', 'age', 'apacheadmissiondx', 'admissionheight', 'hospitaladmittime24', 'hospitaladmitsource', 'admissionweight']]

### Treatments

In [64]:
def getOneHotConditions(aDf, aColumn, aPrefix):
    aDf['conditions'] = aDf[aColumn].str.split('|')
    one_hot = aDf['conditions'].explode().str.get_dummies().groupby(level=0).sum()
    one_hot = one_hot.add_prefix(aPrefix)
    aDf = aDf.drop(columns=['conditions']).join(one_hot)
    return aDf, one_hot

In [65]:
treatment_df = pd.read_csv(database_folder+'treatment.csv')
treatment_df = treatment_df[treatment_df['patientunitstayid'].isin(myIds[0])]

In [66]:
treatment_df, one_hot = getOneHotConditions(treatment_df, 'treatmentstring', 'treatment_')

In [67]:
filtered_df = treatment_df[(treatment_df.treatmentoffset < myHours)].groupby('patientunitstayid').agg('sum').reset_index()
filtered_df_hyp = treatment_df.groupby('patientunitstayid').agg('sum').reset_index()

/tmp/ipykernel_233/1726350892.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  filtered_df = treatment_df[(treatment_df.treatmentoffset < myHours)].groupby('patientunitstayid').agg('sum').reset_index()
/tmp/ipykernel_233/1726350892.py:2: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  filtered_df_hyp = treatment_df.groupby('patientunitstayid').agg('sum').reset_index()


In [68]:
filtered_df[one_hot.columns] = (filtered_df[one_hot.columns] != 0).astype(int)
filtered_df_hyp['hyp'] = (filtered_df_hyp.treatment_hypothermia != 0).astype(int)
filtered_df_hyp.hyp.sum()

603

In [69]:
myPredictorsDf1 = myPredictorsDf.merge(filtered_df[['patientunitstayid'] + list(one_hot.columns)], on=['patientunitstayid'], how='left')
myPredictorsDf1 = myPredictorsDf1.merge(filtered_df_hyp[['patientunitstayid', 'hyp']], on=['patientunitstayid'], how='left')
myPredictorsDf1['treatment_hypothermia'] = myPredictorsDf1.hyp
myPredictorsDf1.drop(columns=['hyp'], inplace = True)

In [70]:
myPredictorsDf = myPredictorsDf1

### Diagnosis

In [71]:
diagnosis_df = pd.read_csv(database_folder+'diagnosis.csv')
diagnosis_df = diagnosis_df[diagnosis_df['patientunitstayid'].isin(myIds[0])]

In [72]:
diagnosis_df, one_hot = getOneHotConditions(diagnosis_df, 'diagnosisstring', 'diagnosis_')

In [ ]:
filtered_df = diagnosis_df[(diagnosis_df.diagnosisoffset < myHours)].groupby('patientunitstayid').agg('sum').reset_index()
filtered_df[one_hot.columns] = (filtered_df[one_hot.columns] != 0).astype(int)

/tmp/ipykernel_233/3375727360.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  filtered_df = diagnosis_df[(diagnosis_df.diagnosisoffset < myHours)].groupby('patientunitstayid').agg('sum').reset_index()


In [ ]:
myPredictorsDf1 = myPredictorsDf.merge(filtered_df[['patientunitstayid'] + list(one_hot.columns)], on=['patientunitstayid'], how='left')

In [ ]:
myPredictorsDf = myPredictorsDf1

### Nurse Charting

In [ ]:
file_path = database_folder+'nurseCharting.csv'
chunk_size = 1e6

df_chunks = []

num_chunks = 0
for chunk in pd.read_csv(file_path,chunksize=chunk_size):
    filtered_chunk = chunk[chunk['patientunitstayid'].isin(myIds[0])]
    df_chunks.append(filtered_chunk)
    if num_chunks % 10 == 0:
        print(f'Chunk {num_chunks} Processed.')
    num_chunks += 1
    del chunk

nurse_charting_df = pd.concat(df_chunks, ignore_index=True)
print('Processing finished.')

### Hypothermmia Analysis

In [ ]:
myDfTempC = nurse_charting_df[(nurse_charting_df.nursingchartcelltypevalname == 'Temperature (C)')]
myDfTempC.nursingchartvalue = myDfTempC.nursingchartvalue.astype(float)
myDfTempC.sort_values(by=['patientunitstayid', 'nursingchartentryoffset'], inplace=True)
# myDfTempC['TimeDiff'] = myDfTempC['nursingchartoffset'].diff()
# myDfTempC['TempTimesTime'] = myDfTempC['TimeDiff'] * myDfTempC['nursingchartvalue'].shift(1)
# myDfTempC['RollingTemp'] = myDfTempC['TempTimesTime'].rolling(window=2).sum()
# myDfTempC['RollingTime'] = myDfTempC['TimeDiff'].rolling(window=2).sum()
# myDfTempC['AvgTemp'] = myDfTempC['RollingTemp'] / myDfTempC['RollingTime']
# myDfTempC[['TempTimesTime', 'TimeDiff', 'nursingchartvalue', 'RollingTemp', 'RollingTime', 'AvgTemp']]
myDfTempC2 = myDfTempC[(myDfTempC.nursingchartentryoffset < 2880)  \
        & (myDfTempC.nursingchartentryoffset > 0)
        & (myDfTempC.nursingchartvalue < 36)]
myDfTempC2['nursingchartentryoffset2'] = myDfTempC2['nursingchartentryoffset']
myHyp = myDfTempC2.groupby('patientunitstayid').agg({'patientunitstayid':'count', 'nursingchartvalue': 'min', 'nursingchartentryoffset':'min', 'nursingchartentryoffset2':'max'})
myHyp['Hyp'] = (myHyp['patientunitstayid'] > 12).astype(int) \
                & (myHyp['nursingchartvalue'] > 25).astype(int) \
                & (myHyp['nursingchartentryoffset2'] - myHyp['nursingchartentryoffset'] > 720).astype(int)
myHyp[(myHyp['Hyp'] == 1)] 

In [ ]:
myPredictorsDf['Hypothermia'] = 0
myPredictorsDf.loc[myPredictorsDf.patientunitstayid.isin(myHyp[myHyp['Hyp'] == 1].index), 'Hypothermia'] = 1
myPredictorsDf[['Hypothermia', 'patientunitstayid']]
myPredictorsDf.Hypothermia.sum()
# myPredictorsDf.drop(columns='Hypothermia', inplace=True)

In [ ]:
myPredictorsDf['both_hypothermia'] = ((myPredictorsDf.treatment_hypothermia == 1) | (myPredictorsDf.Hypothermia == 1)).astype(int)

In [ ]:
(myDfTempC[(myDfTempC.patientunitstayid ==3243869) & (myDfTempC.nursingchartentryoffset < 500)])

### Getting Features 

In [ ]:
def getFeaturesFromDf(aDf, aTimeColumn, aTypeColumn, aValueColumn):
    aDf['num_values'] = pd.to_numeric(aDf[aValueColumn], errors='coerce')
    myMasterGroupDf = aDf[(aDf[aTimeColumn] < myHours)].groupby(['patientunitstayid', aTypeColumn])
    myMasterGroupBegin = aDf.loc[myMasterGroupDf[aTimeColumn].idxmin()]
    myMasterGroupEnd = aDf.loc[myMasterGroupDf[aTimeColumn].idxmax()]
    myMasterGroupAgg = myMasterGroupDf.agg({'mean', 'min', 'max'})
    myMasterGroupAgg = myMasterGroupAgg.num_values.reset_index()
    return myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg

In [ ]:
def mergeFeaturesInDf(aDfToMerge, aBegin, aEnd, aAgg, aPrefix, aTypeColumn, aValueColumn):
    print('Running Begin Columns')
    total = int(aBegin[aTypeColumn].nunique() / 10)
    myPredictorsDf1 = aDfToMerge
    i = 0
    print('progress: ', end="")
    for value in aBegin[aTypeColumn].unique():
        myDf = aBegin[aBegin[aTypeColumn] == value]
        myPredictorsDf1 = myPredictorsDf1.merge(myDf[['patientunitstayid', aValueColumn]], on='patientunitstayid', how='left')
        myPredictorsDf1[aPrefix + '_first_' + value] = myPredictorsDf1[aValueColumn]
        myPredictorsDf1.drop(columns=[aValueColumn], inplace=True)
        if (i % total == 0):
            print('#', end="")
        i+= 1
    print()
    print('Running End Columns')
    total = int(aEnd[aTypeColumn].nunique() / 10)
    i = 0
    print('progress: ', end="")
    for value in aEnd[aTypeColumn].unique():
        myDf = aEnd[aEnd[aTypeColumn] == value]
        myPredictorsDf1 = myPredictorsDf1.merge(myDf[['patientunitstayid', aValueColumn]], on='patientunitstayid', how='left')
        myPredictorsDf1[aPrefix + '_last_' + value] = myPredictorsDf1[aValueColumn]
        myPredictorsDf1.drop(columns=[aValueColumn], inplace=True)
        if (i % total == 0):
            print('#', end="")
        i+= 1
    print()
    print('Running Agg Columns')
    total = int(aEnd[aTypeColumn].nunique() / 10)
    i = 0
    print('progress: ', end="")
    for value in aAgg[aTypeColumn].unique():
        myDf = aAgg[aAgg[aTypeColumn] == value]
        myPredictorsDf1 = myPredictorsDf1.merge(myDf[['patientunitstayid', 'max', 'min', 'mean']], on='patientunitstayid', how='left')
        myPredictorsDf1[aPrefix + '_max_' + value] = myPredictorsDf1['max']
        myPredictorsDf1[aPrefix + '_min_' + value] = myPredictorsDf1['min']
        myPredictorsDf1[aPrefix + '_mean_' + value] = myPredictorsDf1['mean']
        myPredictorsDf1.drop(columns=['max', 'min', 'mean'], inplace=True)
        if (i % total == 0):
            print('#', end="")
        i+= 1
    return myPredictorsDf1

In [ ]:
myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg = getFeaturesFromDf(nurse_charting_df, 'nursingchartoffset', 'nursingchartcelltypevalname', 'nursingchartvalue')

In [ ]:
myPredictorsDf1 = mergeFeaturesInDf(myPredictorsDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg, 'nurse', 'nursingchartcelltypevalname', 'nursingchartvalue')

In [ ]:
print(list(myPredictorsDf.columns))

In [ ]:
myPredictorsDf = myPredictorsDf1

In [ ]:
gcs_df = nurse_charting_df[nurse_charting_df['nursingchartcelltypevalname']=='GCS Total']

In [ ]:
motor_gcs_df = nurse_charting_df[nurse_charting_df.nursingchartcelltypevalname == 'Motor']

In [ ]:
myGcsDf = gcs_df.sort_values(by=['patientunitstayid', "nursingchartoffset"])
myGcsDf['nursingchartoffset2'] = myGcsDf.nursingchartoffset
myGcsDf2 = myGcsDf.groupby('patientunitstayid').agg({'nursingchartoffset':'min', 'nursingchartoffset2': 'max'})
myGcsDfMin = myGcsDf.merge(myGcsDf2, on=['patientunitstayid', 'nursingchartoffset'], how='inner')
myGcsDfMax = myGcsDf.merge(myGcsDf2, on=['patientunitstayid', 'nursingchartoffset2'], how='inner')

In [ ]:
myMGcsDf = motor_gcs_df.sort_values(by=['patientunitstayid', 'nursingchartoffset'])
myMGcsDf['nursingchartoffset2'] = myMGcsDf.nursingchartoffset
myMGcsDf2 = myMGcsDf.groupby('patientunitstayid').agg({'nursingchartoffset':'min', 'nursingchartoffset2': 'max'})
myMGcsDfMin = myMGcsDf.merge(myMGcsDf2, on=['patientunitstayid', 'nursingchartoffset'], how='inner')
myMGcsDfMax = myMGcsDf.merge(myMGcsDf2, on=['patientunitstayid', 'nursingchartoffset2'], how='inner')

In [ ]:
myMGcsDfMax

In [ ]:
myPredictorsDf = myPredictorsDf.merge(myGcsDfMin[['patientunitstayid', 'nursingchartentryoffset', 'nursingchartvalue']], on='patientunitstayid')
myPredictorsDf.rename(columns={'nursingchartentryoffset': 'FirstGCSTime', 'nursingchartvalue': 'FirstGCS'}, inplace=True)
myPredictorsDf =myPredictorsDf.merge(myGcsDfMax[['patientunitstayid', 'nursingchartentryoffset', 'nursingchartvalue']], on='patientunitstayid')
myPredictorsDf.rename(columns={'nursingchartentryoffset': 'LastGCSTime', 'nursingchartvalue': 'LastGCS'}, inplace=True)

In [ ]:
myPredictorsDf = myPredictorsDf.merge(myMGcsDfMin[['patientunitstayid', 'nursingchartentryoffset', 'nursingchartvalue']], on='patientunitstayid', how='outer')
myPredictorsDf.rename(columns={'nursingchartentryoffset': 'FirstMGCSTime', 'nursingchartvalue': 'FirstMGCS'}, inplace=True)
myPredictorsDf =myPredictorsDf.merge(myMGcsDfMax[['patientunitstayid', 'nursingchartentryoffset', 'nursingchartvalue']], on='patientunitstayid', how='outer')
myPredictorsDf.rename(columns={'nursingchartentryoffset': 'LastMGCSTime', 'nursingchartvalue': 'LastMGCS'}, inplace=True)

### Labs

In [ ]:
file_path = database_folder+'lab.csv'
chunk_size = 1e6

# patient_unit_stay_ids = ca_patients_df['patientunitstayid']
df_chunks = []

num_chunks = 0
for chunk in pd.read_csv(file_path,chunksize=chunk_size):
    filtered_chunk = chunk[chunk['patientunitstayid'].isin(myIds[0])]
    df_chunks.append(filtered_chunk)
    if num_chunks % 10 == 0:
        print(f'Chunk {num_chunks} Processed.')
    num_chunks += 1
    del chunk

myLabs = pd.concat(df_chunks, ignore_index=True)
print('Processing finished.')

In [ ]:
myLabs

In [ ]:
myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg = getFeaturesFromDf(myLabs, 'labresultoffset', 'labname', 'labresult')

In [ ]:
myPredictorsDf1 = mergeFeaturesInDf(myPredictorsDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg, 'lab', 'labname', 'labresult')

In [ ]:
myPredictorsDf = myPredictorsDf1

In [ ]:
myPredictorsDf = myPredictorsDf.merge(patients_df[['patientunitstayid', 'hospitaldischargestatus']], on=['patientunitstayid'])

In [ ]:
myPredictorsDf['DeathAtDischarge'] = myPredictorsDf.hospitaldischargestatus == 'Expired'
myPredictorsDf['DeathAtDischarge'] = myPredictorsDf['DeathAtDischarge'].astype(int)

In [ ]:
myPredictorsDf['LastGCS15'] = (myPredictorsDf['LastGCS'] == '15').astype(int)

In [ ]:
myPredictorsDf.loc[myPredictorsDf.age == '> 89', 'age'] = 89

In [ ]:
myPredictorsDf.age = myPredictorsDf.age.astype(int)

In [ ]:
myPredictorsDf.to_csv('eICUPredictorsDiag2.csv', index=False)